In [ ]:
import tensorflow as tf
import sklearn
from keras.applications import EfficientNetV2B0, ResNet50, VGG16
from keras.optimizers import Adam
from keras.preprocessing import image_dataset_from_directory
from keras.losses import CategoricalCrossentropy, BinaryCrossentropy
import kagglehub
import os
from sklearn.model_selection import train_test_split
from datasets import load_dataset

In [ ]:
# ds = load_dataset("ILSVRC/imagenet-1k")
pretrained_imagenet = load_dataset("timm/mini-imagenet")

In [ ]:
LEARNING_RATE=0.0001
BATCH_SIZE=64
IMAGE_SIZE=224
VAL_SPLIT=0.2

In [ ]:
efficientnetv2b0 = EfficientNetV2B0(weights='imagenet', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), include_top=False)
resnet50 = ResNet50(weights='imagenet', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), include_top=False)
vgg16 = VGG16(weights='imagenet', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), include_top=False)

In [ ]:
dataset_path = kagglehub.dataset_download("doctorstrange420/real-and-fake-ai-generated-art-images-dataset")
dataset_path

In [ ]:
os.listdir(dataset_path+"\Data")

In [ ]:
fake_dir = os.path.join(dataset_path+"\Data", "FAKE")
fake_dir

In [ ]:
real_dir = os.path.join(dataset_path+"/Data", "REAL")
real_dir

In [ ]:
os.listdir(real_dir)[:5]

In [ ]:
os.listdir(fake_dir)[:5]

In [ ]:
trainingDs = image_dataset_from_directory(dataset_path+"\Data", seed=123, subset="training", validation_split=VAL_SPLIT, batch_size=BATCH_SIZE, image_size=(IMAGE_SIZE, IMAGE_SIZE))
trainingDs

In [ ]:
valDs = image_dataset_from_directory(dataset_path+"\Data", seed=123, subset="validation", validation_split=VAL_SPLIT, batch_size=BATCH_SIZE, image_size=(IMAGE_SIZE, IMAGE_SIZE))
valDs

In [ ]:
vgg16.trainable=True

In [ ]:
x = vgg16.output
x = tf.keras.layers.Flatten()(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.5)(x)
output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

In [ ]:
vgg16 = tf.keras.Model(inputs=vgg16.input, outputs=output)

In [ ]:
vgg16.summary()

In [ ]:
vgg16.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss=BinaryCrossentropy(), 
              metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

In [ ]:
history = vgg16.fit(
    trainingDs,
    validation_data=valDs,
    epochs=10
)